In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load Data

In [3]:
# ==========================================
# 1. TRANSFORMATIONS
# ==========================================

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.80, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
# ==========================================
# 2. LOAD DATASETS
# ==========================================

dataset_path = './streamlit-app/dataset'

# Dataset without transformations, for obtaining labels
base_dataset = datasets.ImageFolder(root=dataset_path)

# Separate datasets with their respective transformations
train_full = datasets.ImageFolder(
    root=dataset_path,
    transform=train_transforms
)

val_full = datasets.ImageFolder(
    root=dataset_path,
    transform=val_transforms
)

In [5]:
# ==========================================
# 3. STRATIFIED TRAIN-VALIDATION SPLIT
# ==========================================

indices = list(range(len(base_dataset)))
labels = base_dataset.targets

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.25,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)

In [6]:
# ==========================================
# 4. DATA LOADERS
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    pin_memory=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True,
    num_workers=2
)

In [7]:
# ==========================================
# 5. DATASET INFORMATION
# ==========================================

classes = base_dataset.classes
num_classes = len(classes)

print("Classes:", classes)
print("Number of classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Classes: ['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']
Number of classes: 6
Training images: 1779
Validation images: 594


## Model 5: EfficientNet V2-S with Hyper parameter tuning

In [8]:
class CarClassifierEffNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.29):
        super().__init__()
        self.model = models.efficientnet_v2_s(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False

        # Retrieve the in_features of the original classifier (which is a Sequential block)
        in_features = self.model.classifier[1].in_features
        # Unfreeze top 2 layers and fc layers
        for param in self.model.features[-2:].parameters():
            param.requires_grad = True
        # Replace the final fully connected layer
        self.model.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [9]:
def objective(trial):
    # Suggest values for the hyperparameters
    lr_bb = trial.suggest_float('lr_bb', 1e-6, 1e-4, log=True)
    lr_cl = trial.suggest_float('lr_cl', 1e-4, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    # load the model
    model = CarClassifierEffNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([
        {'params': model.model.features[-2:].parameters(), 'lr_bb': lr_bb},
        {'params': model.model.classifier.parameters(), 'lr_cl': lr_cl}
    ], weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.2)
    scaler = torch.amp.GradScaler('cuda')

    # training the loop
    epochs = 15
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_num, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            # Backward pass
            scaler.scale(loss).backward()
            # Update the weights
            scaler.step(optimizer)
            scaler.update()

            if (batch_num+1) % 10 == 0:
                print(f"Batch: {batch_num+1}, Epoch: {epoch+1}, Loss: {loss.item():.2f}")

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss/len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

    # Model Evaluation:

        model.eval()
        total = 0.0
        correct = 0.0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct / total

        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        # FIXED: Call the scheduler step at the end of the epoch using validation loss
        scheduler.step(avg_val_loss)

    return val_accuracy

In [10]:
import optuna

# 1. Add a pruner to kill bad hyperparameter trials early
# startup_trials=5: Allows the first 5 trials to run fully to establish a baseline performance median
# n_warmup_steps=3: Gives trials 3 epochs to warm up before they are eligible for pruning
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)

# 2. Run the optimization
study.optimize(objective, n_trials=20)

# 3. Print the breakthrough results
print("Best trial visual baseline accuracy:")
print(f"  Value: {study.best_trial.value:.2f}%")
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-11-09 19:31:15,054] A new study created in memory with name: no-name-847df92e-4010-40e4-9754-93d93390faec


Batch: 10, Epoch: 1, Loss: 1.30
Batch: 20, Epoch: 1, Loss: 1.05
Batch: 30, Epoch: 1, Loss: 0.96
Batch: 40, Epoch: 1, Loss: 0.79
Batch: 50, Epoch: 1, Loss: 0.73
Epoch [1/15], Loss: 1.0556
Batch: 10, Epoch: 2, Loss: 0.51
Batch: 20, Epoch: 2, Loss: 0.94
Batch: 30, Epoch: 2, Loss: 0.75
Batch: 40, Epoch: 2, Loss: 0.61
Batch: 50, Epoch: 2, Loss: 0.75
Epoch [2/15], Loss: 0.6849
Batch: 10, Epoch: 3, Loss: 0.32
Batch: 20, Epoch: 3, Loss: 0.60
Batch: 30, Epoch: 3, Loss: 0.47
Batch: 40, Epoch: 3, Loss: 0.60
Batch: 50, Epoch: 3, Loss: 0.33
Epoch [3/15], Loss: 0.5731
Batch: 10, Epoch: 4, Loss: 0.74
Batch: 20, Epoch: 4, Loss: 0.73
Batch: 30, Epoch: 4, Loss: 0.45
Batch: 40, Epoch: 4, Loss: 0.32
Batch: 50, Epoch: 4, Loss: 0.64
Epoch [4/15], Loss: 0.4999
Batch: 10, Epoch: 5, Loss: 0.38
Batch: 20, Epoch: 5, Loss: 0.41
Batch: 30, Epoch: 5, Loss: 0.54
Batch: 40, Epoch: 5, Loss: 0.28
Batch: 50, Epoch: 5, Loss: 0.46
Epoch [5/15], Loss: 0.3675
Batch: 10, Epoch: 6, Loss: 0.45
Batch: 20, Epoch: 6, Loss: 0.23
B

[I 2026-11-09 19:40:50,064] Trial 0 finished with value: 84.68013468013469 and parameters: {'lr_bb': 5.611516415334504e-06, 'lr_cl': 0.0008927180304353628, 'dropout_rate': 0.5659969709057024, 'weight_decay': 6.251373574521755e-05}. Best is trial 0 with value: 84.68013468013469.


Batch: 10, Epoch: 1, Loss: 0.95
Batch: 20, Epoch: 1, Loss: 0.99
Batch: 30, Epoch: 1, Loss: 0.85
Batch: 40, Epoch: 1, Loss: 0.67
Batch: 50, Epoch: 1, Loss: 0.86
Epoch [1/15], Loss: 0.9532
Batch: 10, Epoch: 2, Loss: 0.54
Batch: 20, Epoch: 2, Loss: 0.42
Batch: 30, Epoch: 2, Loss: 0.64
Batch: 40, Epoch: 2, Loss: 0.48
Batch: 50, Epoch: 2, Loss: 0.41
Epoch [2/15], Loss: 0.6201
Batch: 10, Epoch: 3, Loss: 0.66
Batch: 20, Epoch: 3, Loss: 0.62
Batch: 30, Epoch: 3, Loss: 0.40
Batch: 40, Epoch: 3, Loss: 0.71
Batch: 50, Epoch: 3, Loss: 0.95
Epoch [3/15], Loss: 0.5476
Batch: 10, Epoch: 4, Loss: 0.48
Batch: 20, Epoch: 4, Loss: 0.29
Batch: 30, Epoch: 4, Loss: 0.93
Batch: 40, Epoch: 4, Loss: 0.44
Batch: 50, Epoch: 4, Loss: 0.49
Epoch [4/15], Loss: 0.4857
Batch: 10, Epoch: 5, Loss: 0.25
Batch: 20, Epoch: 5, Loss: 0.51
Batch: 30, Epoch: 5, Loss: 0.40
Batch: 40, Epoch: 5, Loss: 0.46
Batch: 50, Epoch: 5, Loss: 0.54
Epoch [5/15], Loss: 0.3748
Batch: 10, Epoch: 6, Loss: 0.36
Batch: 20, Epoch: 6, Loss: 0.26
B

[I 2026-11-09 19:50:13,644] Trial 1 finished with value: 84.68013468013469 and parameters: {'lr_bb': 2.05133826308745e-06, 'lr_cl': 0.00014321698289111514, 'dropout_rate': 0.22904180608409974, 'weight_decay': 0.0003967605077052988}. Best is trial 0 with value: 84.68013468013469.


Batch: 10, Epoch: 1, Loss: 1.12
Batch: 20, Epoch: 1, Loss: 0.91
Batch: 30, Epoch: 1, Loss: 0.92
Batch: 40, Epoch: 1, Loss: 0.69
Batch: 50, Epoch: 1, Loss: 0.94
Epoch [1/15], Loss: 0.9937
Batch: 10, Epoch: 2, Loss: 0.62
Batch: 20, Epoch: 2, Loss: 0.68
Batch: 30, Epoch: 2, Loss: 0.44
Batch: 40, Epoch: 2, Loss: 0.38
Batch: 50, Epoch: 2, Loss: 0.73
Epoch [2/15], Loss: 0.6531
Batch: 10, Epoch: 3, Loss: 0.40
Batch: 20, Epoch: 3, Loss: 0.48
Batch: 30, Epoch: 3, Loss: 0.83
Batch: 40, Epoch: 3, Loss: 0.48
Batch: 50, Epoch: 3, Loss: 0.46
Epoch [3/15], Loss: 0.5294
Batch: 10, Epoch: 4, Loss: 0.54
Batch: 20, Epoch: 4, Loss: 0.34
Batch: 30, Epoch: 4, Loss: 0.43
Batch: 40, Epoch: 4, Loss: 0.40
Batch: 50, Epoch: 4, Loss: 0.57
Epoch [4/15], Loss: 0.4502
Batch: 10, Epoch: 5, Loss: 0.27
Batch: 20, Epoch: 5, Loss: 0.27
Batch: 30, Epoch: 5, Loss: 0.40
Batch: 40, Epoch: 5, Loss: 0.45
Batch: 50, Epoch: 5, Loss: 0.42
Epoch [5/15], Loss: 0.3577
Batch: 10, Epoch: 6, Loss: 0.15
Batch: 20, Epoch: 6, Loss: 0.29
B

... (32 outputs left out)

[I 2026-11-09 21:13:54,951] Trial 18 pruned. 


Batch: 10, Epoch: 1, Loss: 1.11
Batch: 20, Epoch: 1, Loss: 1.00
Batch: 30, Epoch: 1, Loss: 1.15
Batch: 40, Epoch: 1, Loss: 0.84
Batch: 50, Epoch: 1, Loss: 0.90
Epoch [1/15], Loss: 1.0018
Batch: 10, Epoch: 2, Loss: 0.70
Batch: 20, Epoch: 2, Loss: 0.46
Batch: 30, Epoch: 2, Loss: 0.70
Batch: 40, Epoch: 2, Loss: 0.94
Batch: 50, Epoch: 2, Loss: 0.80
Epoch [2/15], Loss: 0.7143
Batch: 10, Epoch: 3, Loss: 0.65
Batch: 20, Epoch: 3, Loss: 0.39
Batch: 30, Epoch: 3, Loss: 0.68
Batch: 40, Epoch: 3, Loss: 0.66
Batch: 50, Epoch: 3, Loss: 0.80
Epoch [3/15], Loss: 0.5680
Batch: 10, Epoch: 4, Loss: 0.83
Batch: 20, Epoch: 4, Loss: 0.44
Batch: 30, Epoch: 4, Loss: 0.49
Batch: 40, Epoch: 4, Loss: 0.44
Batch: 50, Epoch: 4, Loss: 0.16
Epoch [4/15], Loss: 0.4606


[I 2026-11-09 21:16:18,062] Trial 19 pruned. 


Best trial visual baseline accuracy:
  Value: 86.20%
  Params: 
    lr_bb: 4.0596116104843085e-06
    lr_cl: 0.0003347776308515934
    dropout_rate: 0.4159725093210579
    weight_decay: 7.4763120622522945e-06
